## Setup

In [1]:
from google.colab import userdata
access_token = userdata.get('CASM-NER')

In [2]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets
# !pip install git+https://github.com/ay94/multilingual-ner.git

In [3]:
# from ner import evaluation

In [4]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [5]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### masakhane/masakhaner2

In [6]:
label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6,
    "B-DATE": 7,
    "I-DATE": 8,
}

masakhaner2 = ner.ReadNERData()
masakhaner2_words, masakhaner2_labels = masakhaner2.read_dataset('masakhane/masakhaner2', label_map, lang='zul')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/5848 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/836 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1670 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/1670 [00:00<?, ?it/s]

In [7]:
print(ner.check_labels(masakhaner2_labels))
label_alignment = {
    'I-PER': 'I-PER',
    'I-DATE': 'O',
    'B-ORG': 'B-ORG',
    'B-LOC': 'B-LOC',
    'I-LOC': 'I-LOC',
    'O':     'O',
    'B-DATE': 'O',
    'B-PER': 'B-PER',
    'I-ORG': 'I-ORG',
}

# Align the dataset labels to the standard labels
masakhaner2_labels = ner.align_dataset(masakhaner2_labels, label_alignment)
print(ner.check_labels(masakhaner2_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC

{'B-ORG', 'B-PER', 'I-LOC', 'B-LOC', 'O', 'I-PER', 'B-DATE', 'I-ORG', 'I-DATE'}
{'B-ORG', 'B-PER', 'I-LOC', 'B-LOC', 'O', 'I-PER', 'I-ORG'}


# Evaluate model

In [10]:
alignment = {
'O':'O',
'B-DATE':'O',
'I-DATE':'O',
'B-PER':'B-PER',
'I-PER':'I-PER',
'B-ORG':'B-ORG',
'I-ORG':'I-ORG',
'B-LOC':'B-LOC',
'I-LOC':'I-LOC',
 }

model_name = "masakhane/afroxlmr-large-ner-masakhaner-1.0_2.0"
model_name_output = 'masakhane/afroxlmr-large-ner-masakhaner-1.0_2.0'
model_evaluation = ner.ModelEvaluation(
    model_name,
    alignment
)

In [11]:
model_evaluation.model.config.id2label

{0: 'O',
 1: 'B-DATE',
 2: 'I-DATE',
 3: 'B-PER',
 4: 'I-PER',
 5: 'B-ORG',
 6: 'I-ORG',
 7: 'B-LOC',
 8: 'I-LOC'}

### masakhane/masakhaner2

In [12]:
data_name = "masakhane/masakhaner2"
masakhaner2_evaluation_output = model_evaluation.evaluate_model(masakhaner2_words, masakhaner2_labels)

  0%|          | 0/105 [00:00<?, ?it/s]

In [13]:
masakhaner2_seqeval = masakhaner2_evaluation_output.get_classification('Seqeval')
masakhaner2_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.8883,0.9436,0.9151,337
1,ORG,0.8421,0.8579,0.8499,373
2,PER,0.9512,0.9651,0.9581,888
3,micro,0.9121,0.9355,0.9237,1598
4,macro,0.8938,0.9222,0.9077,1598
5,weighted,0.9124,0.9355,0.9238,1598


In [14]:
masakhaner2_sklearn = masakhaner2_evaluation_output.get_classification('Sklearn')
masakhaner2_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.8989,0.9496,0.9235,337
1,B-ORG,0.8827,0.8874,0.8850,373
2,B-PER,0.9589,0.9730,0.9659,888
3,I-LOC,0.8882,0.9286,0.9079,154
4,I-ORG,0.8314,0.8251,0.8282,263
5,I-PER,0.9807,0.9849,0.9828,464
6,O,0.9964,0.9947,0.9955,23607
7,accuracy,0.9895,26086,None,None
8,macro,0.9196,0.9347,0.9270,26086
9,weighted,0.9897,0.9895,0.9896,26086
